# Training — Autoencoder (AE)

<div style="text-align: justify">

The following notebook trains a symmetric <b>Autoencoder (AE)</b> for the <b>Tau Anomaly Detection</b> analysis. The model is trained on <b>background-only</b> events using PyTorch Lightning. Signal events are held out and used only to visualise reconstruction error separation after training. Experiment metrics are tracked with <b>Weights & Biases</b>.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis and model configuration |
| DataModule | `datamodule.AnomalyDataModule` | Read mc.parquet, split bkg/sig, fit scaler, build dataloaders |
| Model | `ae.Autoencoder` | Instantiate symmetric AE from config |
| Logger | `WandbLogger` | Initialise Weights & Biases experiment tracking |
| Train | `lightning.Trainer` | Fit model with early stopping and checkpointing |
| Scores | `anomaly.reconstruction_error` | Compute per-event anomaly scores on predict set |
| Diagnostics | `plots` | Loss curves, reconstruction error distributions, feature histograms |

The same pipeline is available as a CLI via `uv run python run.py stage=train model=ae`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)

Experiment Tracking:
* [Weights & Biases](https://wandb.ai/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration with the AE model config.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=ae"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]
plots_dir = path / output_paths["plots_dir"] / "ae"

models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

## DataModule

Setting up the `AnomalyDataModule`. It reads the MC parquet, separates background from signal, fits a scaler on the background training split, and builds `TensorDataset` objects that pair features with event weights.

In [ ]:
import lightning as L

L.seed_everything(cfg.seed, workers=True)

In [ ]:
from src.models.datamodule import AnomalyDataModule

background_origins = {
    s["id"]
    for s in cfg.samples.background.samples
    if s["id"] not in set(cfg.samples.background.get("exclude", []))
}
print(f"Background origins: {background_origins}")

dm = AnomalyDataModule(
    mc_path=str(dataframes_dir / "mc.parquet"),
    background_origins=background_origins,
    normalization=cfg.model.normalization,
    val_fraction=cfg.pipeline.val_fraction,
    batch_size=cfg.model.batch_size,
    seed=cfg.seed,
)
dm.setup()
print(f"Features: {dm.n_features}")
print(f"Feature names: {dm.feature_names_}")

## Model

Instantiating the Autoencoder from the typed config. The architecture mirrors the encoder in reverse for the decoder.

In [ ]:
from omegaconf import OmegaConf

from src.models.ae import Autoencoder
from src.models.config import AEConfig

model_params = dict(OmegaConf.to_container(cfg.model, resolve=True))
model_cfg = AEConfig(**model_params)
model = Autoencoder(model_cfg, n_features=dm.n_features)

n_params = sum(p.numel() for p in model.parameters())
print(f"Autoencoder: {n_params:,} parameters")
print(model)

## Training

### WandB Logger

Initialising the Weights & Biases logger. Set `enabled: false` in `configs/pipeline/default.yaml` to disable tracking.

In [ ]:
from lightning.pytorch.loggers import WandbLogger

wandb_cfg = cfg.pipeline.wandb
if wandb_cfg.enabled:
    wandb_logger = WandbLogger(
        project=wandb_cfg.project,
        name=f"{cfg.experiment_name}-ae",
        log_model=wandb_cfg.log_model,
        config=dict(OmegaConf.to_container(cfg.model, resolve=True)),
    )
else:
    wandb_logger = False

print(f"WandB logging: {'enabled' if wandb_cfg.enabled else 'disabled'}")

### Trainer

Creating the Lightning Trainer with early stopping and model checkpointing.

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor=cfg.pipeline.monitor_metric,
        mode=cfg.pipeline.monitor_mode,
        patience=cfg.pipeline.early_stopping_patience,
        verbose=True,
    ),
    ModelCheckpoint(
        dirpath=models_dir,
        filename="ae-best",
        monitor=cfg.pipeline.monitor_metric,
        mode=cfg.pipeline.monitor_mode,
        save_top_k=1,
        verbose=True,
    ),
]

trainer = L.Trainer(
    max_epochs=cfg.model.n_epochs,
    callbacks=callbacks,
    logger=wandb_logger,
    deterministic=True,
    precision="16-mixed" if cfg.model.amp else "32-true",
    enable_progress_bar=True,
)

### Fit

Training the autoencoder on background-only data.

In [ ]:
trainer.fit(model, datamodule=dm)

### Checkpoint

Saving the final checkpoint (includes scaler state from the DataModule).

In [ ]:
best_path = getattr(trainer.checkpoint_callback, "best_model_path", None)
print(f"Best checkpoint: {best_path}")

ckpt_path = models_dir / "ae.ckpt"
trainer.save_checkpoint(ckpt_path)
print(f"Saved checkpoint: {ckpt_path}")

## Diagnostics

### Loss Curves

Plotting train and validation loss over epochs.

In [ ]:
from src.models.plots import plot_loss

metrics = trainer.logged_metrics
train_loss = [m["train_loss"] for m in trainer.callback_metrics_history if "train_loss" in m] if hasattr(trainer, "callback_metrics_history") else []

# Retrieve per-epoch metrics from the CSV logger or WandB
# For notebook display, use the metric_collection from callbacks
if hasattr(trainer, "logged_metrics"):
    print(f"Final train_loss: {metrics.get('train_loss', 'N/A')}")
    print(f"Final val_loss: {metrics.get('val_loss', 'N/A')}")

### Anomaly Scores

Computing reconstruction error on the predict set (background validation + signal). Higher reconstruction error indicates a more anomalous event.

In [ ]:
import torch
import numpy as np

from src.models.anomaly import reconstruction_error, compute_threshold, build_scores_frame

predictions = trainer.predict(model, datamodule=dm)
x_hat = torch.cat(predictions)
x_orig = torch.cat([batch[0] for batch in dm.predict_dataloader()])

scores = reconstruction_error(x_orig, x_hat).numpy()
print(f"Scores: {len(scores):,} events")
print(f"  Background mean: {scores[dm.predict_labels == 0].mean():.6f}")
print(f"  Signal mean:     {scores[dm.predict_labels == 1].mean():.6f}")

### Threshold

In [ ]:
bkg_scores = scores[dm.predict_labels == 0]
sig_scores = scores[dm.predict_labels == 1]

threshold = compute_threshold(
    bkg_scores,
    strategy=cfg.pipeline.threshold_strategy,
    percentile=cfg.pipeline.threshold_percentile,
)
print(f"Threshold ({cfg.pipeline.threshold_strategy}): {threshold:.6f}")
print(f"  Background flagged: {(bkg_scores > threshold).sum():,} / {len(bkg_scores):,}")
print(f"  Signal flagged:     {(sig_scores > threshold).sum():,} / {len(sig_scores):,}")

### Reconstruction Error Distribution

Comparing the reconstruction error distributions of background and signal events.

In [ ]:
from src.models.plots import plot_reconstruction_error

fig = plot_reconstruction_error(bkg_scores, sig_scores, threshold=threshold, title="AE Reconstruction Error")
fig.savefig(plots_dir / "ae_reconstruction_error.png", dpi=150, bbox_inches="tight")

### Single Event Reconstruction

Bar chart comparing original vs reconstructed feature values for a single event.

In [ ]:
from src.models.plots import plot_reconstruction_performance

fig = plot_reconstruction_performance(
    x_orig[0].numpy(),
    x_hat[0].numpy(),
    dm.feature_names_,
    event_idx=0,
    title="AE Single Event Reconstruction",
)
fig.savefig(plots_dir / "ae_single_event.png", dpi=150, bbox_inches="tight")

### Feature Histograms

Per-feature distributions of original vs reconstructed values across the predict set.

In [ ]:
from src.models.plots import plot_feature_histograms

fig = plot_feature_histograms(
    x_orig.numpy(),
    x_hat.numpy(),
    dm.feature_names_,
    title="AE Feature Distributions",
)
fig.savefig(plots_dir / "ae_feature_histograms.png", dpi=150, bbox_inches="tight")

### Scores DataFrame

Building and saving the tidy scores DataFrame for downstream evaluation.

In [ ]:
scores_df = build_scores_frame(scores, dm.predict_labels, dm.predict_origins)
scores_path = dataframes_dir / "ae_scores.parquet"
scores_df.to_parquet(scores_path)
print(f"Saved scores: {scores_path}")
scores_df.head()

### Finish WandB

Closing the WandB run.

In [ ]:
import wandb

if wandb.run is not None:
    wandb.finish()